# Prepare MOSS style training in Colab
Use **Runtime → Change runtime type → GPU**. Setup installs an isolated uv environment and downloads model assets. The later cells load those local pretrained weights and launch a persistent training run with logging and resume. Use the pretrained preset for this run; tiny random models are a separate development configuration.

By default, clone **TheDucker1/sd-scripts**, branch **style_loss**. Commit and push the current MOSS changes there first. All required models are fetched, with ConvRot INT8 Qwen3.5-9B by default and no duplicate BF16 text-encoder download. A successful setup does not establish that full training fits the assigned GPU/RAM.

In [ ]:
import io, json, os, shutil, stat, subprocess, sys, zipfile
from pathlib import Path
if sys.platform != 'linux' or not Path('/content').is_dir():
    raise RuntimeError('This notebook is for Google Colab only; do not run it locally.')
from google.colab import files, userdata
print('Colab guard passed. No installer has run yet.')


In [ ]:
# Default: clone or fast-forward the committed fork. Upload/existing remain optional.
SOURCE = 'git'  # 'git', 'upload', or 'existing'
REPO_URL = 'https://github.com/TheDucker1/sd-scripts'
BRANCH = 'style_loss'
REPO = Path('/content/sd-scripts')
if SOURCE == 'upload':
    uploaded = files.upload()
    archives = [data for name, data in uploaded.items() if name.lower().endswith('.zip')]
    if len(archives) != 1:
        raise RuntimeError('Upload exactly one source-only repository ZIP.')
    destination = Path('/content/moss-upload')
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(archives[0])) as archive:
        if sum(info.file_size for info in archive.infolist()) > 512 * 1024**2:
            raise RuntimeError('Source ZIP exceeds 512 MiB; exclude models, datasets and environments.')
        for info in archive.infolist():
            target = (destination / info.filename).resolve()
            if destination not in target.parents or stat.S_ISLNK(info.external_attr >> 16):
                raise RuntimeError('Unsafe archive entry: ' + info.filename)
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as source, target.open('wb') as output:
                    shutil.copyfileobj(source, output)
    candidates = list(destination.rglob('moss_image_train_style.py'))
    if len(candidates) != 1:
        raise RuntimeError('Expected exactly one current MOSS style trainer in the ZIP.')
    REPO = candidates[0].parent
    del archives, uploaded
elif SOURCE == 'git':
    if not REPO_URL:
        raise RuntimeError('Set REPO_URL.')
    if not REPO.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)
    else:
        def git_output(*args):
            return subprocess.check_output(['git', '-C', str(REPO), *args], text=True).strip()
        if git_output('remote', 'get-url', 'origin').rstrip('/').removesuffix('.git') != REPO_URL.rstrip('/').removesuffix('.git'):
            raise RuntimeError('Existing checkout has a different origin.')
        if git_output('symbolic-ref', '--short', 'HEAD') != BRANCH:
            raise RuntimeError('Existing checkout is on a different branch.')
        if git_output('status', '--porcelain', '--untracked-files=no'):
            raise RuntimeError('Tracked files have local edits. Commit or stash them before updating; no edits were discarded.')
        subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO), 'merge', '--ff-only', 'FETCH_HEAD'], check=True)
    print('Source commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip())
elif SOURCE != 'existing':
    raise ValueError('Unknown SOURCE')
SETUP = REPO / 'tools/colab/setup_moss_style.py'
if not SETUP.is_file():
    raise RuntimeError('Missing setup script; commit and push all new MOSS files to the selected fork/branch first.')
print('Using source:', REPO)


In [ ]:
# One-time authentication for this runtime. Never put your token in code or chat.
# Your HF account must already have access to the gated DINOv3 model.
from getpass import getpass
FORCE_TOKEN_PROMPT = False  # Set True to replace an expired/rejected token.
token = None if FORCE_TOKEN_PROMPT else os.environ.get('HF_TOKEN')
if not token and not FORCE_TOKEN_PROMPT:
    try:
        token = userdata.get('HF_TOKEN')
    except Exception:
        token = None
if not token:
    token = getpass('Hugging Face read token (hidden, this runtime only): ').strip()
if not token or not token.strip():
    raise RuntimeError('An authorized HF token is required for the gated DINOv3 download.')
os.environ['HF_TOKEN'] = token.strip()
print('HF token is available for this runtime. No token value is printed or written to disk.')
del token


In [ ]:
DOWNLOAD_MODELS = True
TEXT_ENCODER = 'int8_convrot'  # Supported profiles: 'int8_convrot' (default), 'bf16'.
# W4A8/GPTQ, GGUF, and arbitrary replacement encoders are not supported here.
PERSIST_MODELS_TO_DRIVE = False
VENV = Path('/content/moss-venv')
MODELS = Path('/content/moss-models')
if PERSIST_MODELS_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    MODELS = Path('/content/drive/MyDrive/moss-models')
print('Environment:', VENV, '\nAssets:', MODELS)


In [ ]:
# This is the only installation/download step. Run it on the Colab VM.
# Setup validates DINOv3 access before downloading any model weights.
command = [sys.executable, str(SETUP), '--repo', str(REPO), '--repo-url', REPO_URL, '--branch', BRANCH, '--venv', str(VENV), '--models-dir', str(MODELS), '--text-encoder', TEXT_ENCODER]
if not DOWNLOAD_MODELS:
    command.append('--skip-models')
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
# Inspect metadata only. These cells do not load checkpoint tensors.
print((VENV / 'moss-runtime.json').read_text())
for name in ['resolved_models.json', 'dataset_stream.json']:
    path = MODELS / name
    if path.exists():
        print(name, '->', path)
print('Use this interpreter for later commands:', VENV / 'bin/python')
print('Preparation only: no training has started.')


The environment is separate from the notebook kernel. Launch later scripts with `/content/moss-venv/bin/python`; shell activation in a previous cell does not persist. `moss-installed-packages.txt` records resolved package versions.

`resolved_models.json` records exact model revisions, local paths, and selected text-encoder loader settings. The default INT8 file is fetched from Comfy-Org/MOSS-Image; full BF16 Qwen weights are excluded. `dataset_stream.json` lists pinned shard URLs, not downloaded images. SigLIP contributes config files and MegaStyle supplies its vision weights. `moss-source.json` in the venv records the code commit.

The cells below load MOSS SFT, the published editing LoRA, INT8/BF16 Qwen, the custom PixAI tagger, DINOv3 and MegaStyle from the resolved local assets. Only the editing LoRA is optimized. GPU/RAM fit and training quality must be assessed on the Colab host.

## Training settings and persistence
Run the following cells only in Colab. Outputs go to Google Drive. Keep the same RUN_NAME and settings to resume; use a new name for a different experiment. Each step is an optimizer update: batch 1 × accumulation 2 = two A/B pairs per step.

| Setting | Default | Meaning |
|---|---|---|
| PIXEL_BUDGET | 768 × 768 | Maximum area **per image**, for both A and B. Aspect buckets use 64-pixel increments; shapes vary and rounding can leave unused area. |
| MAX_STEPS | 100000 | Total optimizer updates, including updates completed before resume. |
| PREVIEW_EVERY / PREVIEW_PAIRS | 1000 / 5 | Five fresh A/B/result JPEG triptychs, each using a full 50-step inference run. |
| SAVE_EVERY / KEEP_LAST | 1000 / 3 | Keep three completed adapter + resume checkpoints. Preview images and logs remain. |
| LEARNING_RATE / WARMUP_STEPS | 1e-4 / 1000 | AdamW8bit peak LR and initial warmup. |
| LR_CYCLES / MIN_LR_RATIO | 5 / 0.01 | Smooth cosine rises/falls with a linearly decaying envelope, ending at 1e-6. This is a starting preset, not a guarantee of convergence. |

The command uses `--compile --compile_backend inductor --compile_scope blocks --compile_modules dit --compile_dynamic true`: only MOSS transformer blocks compile, with dynamic shapes. Compilation can pause initial progress. Checkpoints exclude frozen model weights; `training_state.pt` stores optimizer, scheduler and RNG. Resume restarts the dataset stream without replaying earlier batches. A bounded local shuffle window holds 16 **resized** images before A/B pairing; set SHUFFLE_BUFFER=0 for no look-ahead. This is synchronous local shuffling, not worker prefetch or asynchronous download. Drive writes can be slow; only completed checkpoints are auto-selected.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_NAME = 'moss-style-768'
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 2
PIXEL_BUDGET = 768 * 768
SHUFFLE_BUFFER = 16  # Processed images, not pairs. Roughly 108 MiB at this pixel budget.
MAX_STEPS = 100_000
PREVIEW_EVERY, PREVIEW_PAIRS, PREVIEW_STEPS = 1000, 5, 50
SAVE_EVERY, KEEP_LAST = 1000, 3
LEARNING_RATE, WARMUP_STEPS = 1e-4, 1000
LR_CYCLES, MIN_LR_RATIO = 5, 0.01
COMPILE = True
OUTPUT_DIR = Path('/content/drive/MyDrive/moss-training') / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
config = json.loads((REPO / 'configs/moss_style_colab.json').read_text())
config['assets_manifest'] = str(MODELS / 'resolved_models.json')
shards = json.loads((MODELS / 'dataset_stream.json').read_text())
config['data'] = {'datasets': [{'batch_size': BATCH_SIZE, 'resolution': [768, 768], 'enable_bucket': True, 'max_pixels': PIXEL_BUDGET, 'min_bucket_reso': 256, 'max_bucket_reso': 1536, 'bucket_reso_steps': 64, 'bucket_no_upscale': False, 'max_aspect_ratio': 3.0, 'repeat': True, 'shuffle_buffer': SHUFFLE_BUFFER, 'shuffle_caption': True, 'resize_interpolation': 'area', 'subsets': [{'shards': shards['shards']}]}]}
config['training'].update(batch_size=BATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION, max_steps=MAX_STEPS, save_every=SAVE_EVERY, keep_last_checkpoints=KEEP_LAST, learning_rate=LEARNING_RATE, warmup_steps=WARMUP_STEPS, lr_cycles=LR_CYCLES, min_lr_ratio=MIN_LR_RATIO)
config['sampling'].update(every_n_steps=PREVIEW_EVERY, num_pairs=PREVIEW_PAIRS, steps=PREVIEW_STEPS)
RUN_CONFIG = OUTPUT_DIR / 'launch_config.json'
if RUN_CONFIG.exists() and json.loads(RUN_CONFIG.read_text()) != config:
    raise RuntimeError('Existing run settings differ. Restore the settings or choose a new RUN_NAME.')
RUN_CONFIG.write_text(json.dumps(config, indent=2) + '\n')
print('Persistent run:', OUTPUT_DIR)
print(json.dumps(config['training'], indent=2))


Run the TensorBoard cell before launching training to keep its dashboard visible while the next cell runs. Losses, learning rate and gradient norm are also written to `metrics.jsonl`; console output goes to `console.log`. JPEG previews are under `samples/step-XXXXXX/` on Drive. Colab runtime limits still apply; this notebook does not keep a disconnected runtime alive.

In [ ]:
TB_LOGDIR = str(OUTPUT_DIR / 'tensorboard')
%load_ext tensorboard
%tensorboard --logdir $TB_LOGDIR


In [ ]:
# Load the downloaded pretrained components and train the editing LoRA.
# Missing/incompatible checkpoints raise an error; there is no random fallback.
import re, shlex
AUTO_RESUME = True
checkpoints = sorted((p for p in OUTPUT_DIR.iterdir() if re.fullmatch(r'step-\d+', p.name) and all((p / name).is_file() for name in ['complete.json', 'adapter_model.safetensors', 'adapter_config.json', 'training_state.pt'])), key=lambda p: int(p.name.split('-')[1]))
command = [str(VENV / 'bin/python'), '-u', str(REPO / 'moss_image_train_style.py'), '--config', str(RUN_CONFIG), '--output_dir', str(OUTPUT_DIR), '--device', 'cuda']
if COMPILE:
    command += ['--compile', '--compile_backend', 'inductor', '--compile_scope', 'blocks', '--compile_modules', 'dit', '--compile_dynamic', 'true']
if AUTO_RESUME and checkpoints:
    command += ['--resume', str(checkpoints[-1])]
elif checkpoints:
    raise RuntimeError('Existing checkpoints found. Enable AUTO_RESUME or choose a new RUN_NAME.')
print(shlex.join(command), flush=True)
with (OUTPUT_DIR / 'console.log').open('a', buffering=1) as log:
    log.write('\nCOMMAND: ' + shlex.join(command) + '\n')
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
        if process.wait():
            raise RuntimeError('Training failed. Read the output above and console.log on Drive.')
    finally:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        process.stdout.close()
